# **Hands-On Lab: Segmentasi Cacat Manufaktur Menggunakan Clustering**
---
* **Mata Kuliah:** Machine Learning (S1)
* **Metodologi:** CRISP-DM (*Business Understanding, Data Understanding, Preprocessing, Modeling, Evaluation*)
* **Algoritma:** K-Means & Hierarchical Clustering (Agglomerative)
* **Dataset:** `defects_data.csv` (Manufacturing Defects Data)

---

## **1. Business Understanding (Pemahaman Bisnis)**
### **Konteks Bisnis**
Dalam operasional industri manufaktur modern, menjaga kualitas produk akhir merupakan prioritas utama demi mempertahankan reputasi merek dan menekan kerugian finansial. Kemunculan cacat produksi (*defects*) tidak hanya mengurangi efisiensi tetapi juga memicu pembengkakan biaya akibat aktivitas pengerjaan ulang maupun perbaikan produk (*repair cost*).

Tim *Quality Assurance* (QA) seringkali dihadapkan pada ribuan log laporan cacat setiap bulannya. Tanpa pemetaan yang jelas, manajemen sulit menentukan lini produksi atau jenis cacat mana yang harus diintervensi terlebih dahulu secara taktis maupun strategis.

### **Tujuan Bisnis**
Mengelompokkan data cacat manufaktur secara otomatis berdasarkan kemiripan karakteristiknya (seperti kombinasi tingkat keparahan cacat dan besarnya pengeluaran finansial untuk perbaikan). Hasil pengelompokkan ini akan digunakan oleh manajemen untuk melakukan **prioritasan perbaikan proses** pada segmen cacat yang paling merugikan keuangan perusahaan atau yang paling sering terjadi.

### **Tujuan Machine Learning**
Menerapkan teknik *Unsupervised Learning* dengan menggunakan dua metode klasterisasi utama:
1. **K-Means Clustering** (Pendekatan berbasis partisi/titik pusat pusat massa/*centroid*)
2. **Hierarchical Clustering** (Pendekatan berbasis hierarki/pohon keputusan dari bawah ke atas/*agglomerative*)

Serta menggunakan alat bantu analitis **Elbow Method** dan **Dendrogram** untuk mendalami landasan matematis dalam penentuan jumlah kelompok (*cluster*) terbaik tanpa adanya label historis.

## **2. Data Understanding & Preprocessing (Pemahaman Data & Pra-pemrosesan)**
### **Deskripsi Variabel Dataset**
* `defect_id`: ID unik penanda rekaman cacat.
* `product_id`: ID produk yang terpengaruh oleh cacat.
* `defect_type`: Jenis cacat secara fisik (*Cosmetic, Functional, Structural*).
* `defect_date`: Tanggal ditemukannya cacat.
* `defect_location`: Lokasi spesifik pada produk (*Surface, Component, Internal*).
* `severity`: Tingkat keparahan cacat (*Minor, Moderate, Critical*).
* `inspection_method`: Metode pendeteksian (*Visual Inspection, Manual Testing, Automated Testing*).
* `repair_cost`: Biaya finansial nyata yang dikeluarkan untuk memperbaiki produk tersebut ($).

Mari kita mulai dengan mengimpor pustaka (*library*) Python yang diperlukan dan memuat data ke dalam memori.

In [ ]:
# Import library standar untuk manipulasi data dan visualisasi
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import library khusus machine learning scikit-learn & scipy
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
import scipy.cluster.hierarchy as sch

# Konfigurasi estetika plot
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("[INFO] Semua library berhasil di-import.")

In [ ]:
# Memuat berkas dataset
# Pastikan berkas 'defects_data.csv' sudah diunggah ke runtime lingkungan Anda (misal: Google Colab files)
try:
    df = pd.read_csv('defects_data.csv')
    print("--- DATASET BERHASIL DIMUAT ---")
except FileNotFoundError:
    print("[ERROR] File 'defects_data.csv' tidak ditemukan. Silakan unggah file tersebut ke direktori kerja Anda.")

In [ ]:
# Inspeksi dimensi dan tipe data
print("=== INFORMASI STRUKTUR DATA ===")
df.info()

In [ ]:
# Melihat sampel baris teratas
print("=== 5 BARIS PERTAMA DATASET ===")
df.head()

In [ ]:
# Analisis deskriptif awal fitur numerik
print("=== STATISTIK DESKRIPTIF NUMERIK ===")
df.describe()

In [ ]:
# Memeriksa sebaran distribusi kelas pada fitur ordinal kualitatif 'severity'
print("=== DISTRIBUSI KATEGORI SEVERITY ===")
print(df['severity'].value_counts())

print("\n=== DISTRIBUSI KATEGORI DEFECT TYPE ===")
print(df['defect_type'].value_counts())

### **Rekayasa Data (Data Preprocessing)**
Algoritma *clustering* berbasis jarak seperti K-Means dan Hierarchical Clustering menghitung kemiripan antar-objek menggunakan rumus matematika jarak kedekatan koordinat (misalnya Jarak Euclidean). Oleh karena itu, kita perlu melakukan dua langkah penting:
1. **Ordinal Encoding**: Mengubah tingkatan kualitatif string `severity` (*Minor, Moderate, Critical*) menjadi angka logis bertingkat ($1, 2, 3$).
2. **Feature Scaling (Standardization)**: Jangkauan nilai `repair_cost` berada pada rentang ratusan dolar, sedangkan nilai tingkatan keparahan hasil encoding hanya berkisar antara $1$ hingga $3$. Jika tidak disamakan skalanya melalui z-score standardisasi, variansi besar pada `repair_cost` akan mendominasi perhitungan jarak sepenuhnya, membuat fitur keparahan diabaikan oleh model.

In [ ]:
# 1. Mapping Ordinal Data ke bentuk Angka
severity_mapping = {'Minor': 1, 'Moderate': 2, 'Critical': 3}
df['severity_score'] = df['severity'].map(severity_mapping)

# 2. Isolasi Fitur yang Digunakan untuk Segmentasi
X = df[['repair_cost', 'severity_score']]

# 3. Standardisasi Fitur menggunakan StandardScaler (Mean=0, Std=1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("[INFO] Konversi data kualitatif dan penskalaan fitur selesai dilakukan.")
print("Array 5 baris pertama hasil scaling:\n", X_scaled[:5])

## **3. K-Means dengan Elbow Method**
Algoritma K-Means mengharuskan praktisi menentukan parameter jumlah klaster ($K$) di awal sebelum pelatihan model. Untuk menentukan nilai $K$ yang paling optimal secara objektif (bukan tebakan subjektif), kita menerapkan **Elbow Method**.

Metode ini melatih model K-Means berulang kali menggunakan rentang nilai $K$ tertentu (misal $1$ sampai $10$) dan mencatat skor **WCSS** (*Within-Cluster Sum of Squares*) atau inersia pada setiap iterasi. WCSS mengukur total kuadrat jarak antara setiap titik data dengan titik pusat massa (*centroid*) klasternya masing-masing.

In [ ]:
# Inisialisasi list untuk menyimpan nilai WCSS
wcss = []
k_range = range(1, 11)

# Melakukan iterasi pemodelan K-Means dari K=1 hingga K=10
for k in k_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)  # Keterangan: `.inertia_` menyimpan nilai WCSS model

# Visualisasi Grafik Elbow Method
plt.figure(figsize=(9, 5.5))
plt.plot(k_range, wcss, marker='o', linestyle='--', color='darkblue', linewidth=2, markersize=8)
plt.title('Elbow Method untuk Penentuan Jumlah Klaster (K) Optimal', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Jumlah Klaster (K)', fontsize=11)
plt.ylabel('Within-Cluster Sum of Squares (WCSS / Inersia)', fontsize=11)
plt.xticks(k_range)
plt.grid(True, linestyle=':')
plt.show()

### **Interpretasi Hasil Grafik Elbow**
> **Instruksi Analisis Mahasiswa:** Perhatikan bentuk lekukan pada grafik di atas. Seiring bertambahnya jumlah $K$, nilai WCSS pasti akan turun drastis karena titik pusat semakin dekat dengan data individual. Titik optimum dicapai di mana penurunan nilai WCSS melambat secara dramatis dan membentuk sudut patahan yang menyerupai sikut tangan (*elbow*).
>
> Berdasarkan pola sebaran data manufaktur ini, sikut umumnya terdeteksi secara visual pada nilai **$K = 3$**. Kita akan menetapkan nilai ini untuk tahap pemodelan K-Means berikutnya.

## **4. K-Means Clustering**
Setelah menetapkan jumlah klaster terbaik ($K=3$), kita akan mengonfigurasi objek `KMeans` final, melakukan proses pemetaan kelompok, menyimpan label klasternya ke dalam struktur *dataframe* utama, dan membuat visualisasinya.

In [ ]:
# Eksekusi K-Means final dengan nilai K = 3
optimal_k = 3
kmeans_model = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)

# Fit model dan prediksi label klaster
df['cluster_kmeans'] = kmeans_model.fit_predict(X_scaled)

print("[INFO] Proses klasterisasi dengan K-Means selesai.")
print("Jumlah sampel per klaster K-Means:\n", df['cluster_kmeans'].value_counts())

In [ ]:
# Visualisasi Sebaran Klaster K-Means melalui Scatter Plot
plt.figure(figsize=(10, 6.5))
sns.scatterplot(
    x=df['repair_cost'], 
    y=df['severity_score'], 
    hue=df['cluster_kmeans'], 
    palette='Set1', 
    s=90, 
    alpha=0.85, 
    edgecolor='black', 
    linewidth=0.5
)

plt.title(f'Hasil Segmentasi Kasus Cacat Menggunakan K-Means (K={optimal_k})', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Biaya Perbaikan Produk ($ / Repair Cost)', fontsize=11)
plt.ylabel('Tingkat Keparahan Cacat (Severity Score)', fontsize=11)
plt.yticks([1, 2, 3], ['1 (Minor)', '2 (Moderate)', '3 (Critical)'])
plt.legend(title='Klaster K-Means', loc='upper left', frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
# Analisis Profiling Karakteristik Rata-rata Tiap Kelompok K-Means
print("=== KARAKTERITAS STATISTIK RATA-RATA TIAP KLASTER K-MEANS ===")
profil_kmeans = df.groupby('cluster_kmeans')[['repair_cost', 'severity_score']].mean()
print(profil_kmeans)

## **5. Hierarchical Clustering dengan Dendrogram**
Berbeda dengan K-Means yang membagi data langsung secara sekat spasial, *Hierarchical Clustering* membangun pengelompokkan secara bertahap seperti pohon silsilah keluarga. Jenis pendekatan yang umum digunakan adalah *Agglomerative Clustering* (dari bawah ke atas), di mana awalnya setiap rekaman data adalah satu klaster mandiri, kemudian sepasang klaster dengan jarak terdekat digabungkan berulang kali hingga menyatu utuh.

Untuk menganalisis proses penggabungan bertingkat ini dan menentukan jumlah klaster terbaik secara visual, kita wajib memplot **Dendrogram** terlebih dahulu sebelum melatih fungsi model.

In [ ]:
# Membuat visualisasi grafik Dendrogram pohon keputusan hierarki
plt.figure(figsize=(12, 7.5))

# Menggunakan metode penggabungan 'ward' yang meminimalkan varians di dalam klaster yang digabungkan
linkage_matrix = sch.linkage(X_scaled, method='ward')

# Plotting dendrogram
dendrogram = sch.dendrogram(linkage_matrix)

plt.title('Dendrogram Struktur Hierarki Kasus Cacat Manufaktur (Metode Ward)', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Indeks Baris Sampel Data Cacat', fontsize=11)
plt.ylabel('Jarak Ketidakmiripan Euclidean (Linkage Distance)', fontsize=11)
plt.xticks([])  # Label indeks dihapus/disembunyikan agar visualisasi tetap rapi dan tidak padat
plt.show()

### **Interpretasi Analisis Dendrogram**
> **Instruksi Analisis Mahasiswa:** Cara menentukan jumlah kelompok optimal dari Dendrogram adalah dengan mencari **garis vertikal tertinggi yang tidak terpotong** oleh sub-penggabungan horizontal lainnya.
>
> Jika kita menarik sebuah garis potong khayalan secara horizontal melewati area vertikal yang lapang tersebut, kita dapat melihat bahwa diagram pohon ini terbagi secara ideal menjadi **3 cabang utama** (ditandai oleh perbedaan pewarnaan otomatis pada visualisasi di atas). Oleh karena itu, kita akan melatih model Agglomerative dengan parameter `n_clusters=3`.

## **6. Hierarchical Clustering (Agglomerative)**
Mari kita eksekusi algoritma `AgglomerativeClustering` menggunakan basis jumlah klaster yang telah dianalisis melalui struktur Dendrogram sebelumnya, lalu memetakan hasilnya ke scatter plot.

In [ ]:
# Inisialisasi dan pelatihan model Hierarchical Agglomerative Clustering
# Parameter metric='euclidean' digunakan untuk menghitung jarak antar titik, linkage='ward' untuk kriteria penggabungan
hc_model = AgglomerativeClustering(n_clusters=3, metric='euclidean', linkage='ward')

# Menghasilkan label klaster untuk dataset
df['cluster_hierarchy'] = hc_model.fit_predict(X_scaled)

print("[INFO] Proses klasterisasi dengan Hierarchical Clustering selesai.")
print("Jumlah sampel per klaster Hierarki:\n", df['cluster_hierarchy'].value_counts())

In [ ]:
# Visualisasi Sebaran Klaster Hierarchical melalui Scatter Plot
plt.figure(figsize=(10, 6.5))
sns.scatterplot(
    x=df['repair_cost'], 
    y=df['severity_score'], 
    hue=df['cluster_hierarchy'], 
    palette='Set2', 
    s=90, 
    alpha=0.85, 
    edgecolor='black', 
    linewidth=0.5
)

plt.title('Hasil Segmentasi Kasus Cacat Menggunakan Hierarchical Clustering', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Biaya Perbaikan Produk ($ / Repair Cost)', fontsize=11)
plt.ylabel('Tingkat Keparahan Cacat (Severity Score)', fontsize=11)
plt.yticks([1, 2, 3], ['1 (Minor)', '2 (Moderate)', '3 (Critical)'])
plt.legend(title='Klaster Hierarki', loc='upper left', frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
# Analisis Profiling Karakteristik Rata-rata Tiap Kelompok Hierarchical Clustering
print("=== KARAKTERITAS STATISTIK RATA-RATA TIAP KLASTER HIERARKI ===")
profil_hierarchy = df.groupby('cluster_hierarchy')[['repair_cost', 'severity_score']].mean()
print(profil_hierarchy)

---
## **7. Kesimpulan & Pertanyaan Evaluasi Mandiri**

### **Rekomendasi Manajerial Berdasarkan Klaster (Actionable Insights)**
Berdasarkan nilai rata-rata (*profiling*) fitur penyusun klaster baik pada K-Means maupun Hierarchical, mahasiswa diharapkan mampu menerjemahkan angka analitis menjadi segmentasi bisnis nyata, misalnya:
1. **Segmen Risiko Finansial Tinggi (High-Cost Defects):** Klaster yang memiliki karakteristik tingkat keparahan tinggi (*Critical*) dan rata-rata pengeluaran biaya perbaikan paling masif. Manajemen QA wajib menempatkan sistem kendali mutu otomatis (*Automated Testing*) utama pada jalur ini untuk mencegah lolosnya cacat ke tangan konsumen.
2. **Segmen Minoritas Efisien (Low-Cost / Minor Defects):** Klaster dengan kerugian finansial kecil dan tingkat keparahan rendah (*Minor*). Penanganan dapat dilakukan melalui inspeksi berkala reguler tanpa perlu intervensi biaya operasional besar.

### **Tugas Laporan Praktikum (Pertanyaan untuk Mahasiswa)**
Jawablah pertanyaan analisis di bawah ini pada lembar laporan praktikum Anda:
1. **Analisis Komparasi Spasial**: Bandingkan sebaran visual scatter plot dari hasil algoritma K-Means (Langkah 4) dan Hierarchical Clustering (Langkah 6). Apakah batas-batas klasternya terlihat identik atau terdapat pergeseran titik sampel data tertentu? Jelaskan mengapa hal itu bisa terjadi dari sisi perbedaan algoritma dasar keduanya!
2. **Esensi Data Preprocessing**: Mengapa pengerjaan standardisasi fitur (`StandardScaler`) menjadi syarat mutlak yang tidak boleh dilewatkan sebelum menjalankan proses perhitungan pada algoritma berbasis jarak? Apa bahaya analisisnya jika langkah tersebut dilewati?
3. **Kelebihan Metodologis**: Jelaskan menurut analisis Anda pribadi, apa kelebihan analitis visualisasi grafik **Dendrogram** dibandingkan dengan visualisasi **Elbow Method** dalam membantu memetakan pola kedekatan relasi data?